# dim_rls_usuarios — construção

Lê três fontes e grava `lake_gold_fatos.dbo.dim_rls_usuarios`:

| Fonte | Tabela delta | Papel |
|---|---|---|
| Classificação oficial | `lake_gold_fatos.dbo.dim_funcionarios` | base derivada de cargo/secao |
| Grupos estáveis | `lake_prep_siplan.dbo.rls_grupos` | substitui classificação oficial |
| Exceções individuais | `lake_prep_siplan.dbo.rls_overrides` | acrescenta linhas (multi-unidade OK) |

**Precedência:** `rls_grupos` substitui; `rls_overrides` acrescenta.

### Testa como o usuário está na lista de RLS

In [ ]:
# testar o que está em dim_rls_usuarios

display(
    spark.sql("""
        SELECT * FROM lake_gold_fatos.dbo.dim_rls_usuarios
        WHERE email LIKE '%sergio.seabra%'
    """)
)


## Atualiza dim_rls_usuarios

In [4]:
import pandas as pd
from datetime import date

# ── Parâmetros ────────────────────────────────────────────────────────────────
# Ajustar conforme schema real de dim_funcionarios
COL_EMAIL     = 'email'
COL_CARGO     = 'cargo'
COL_SECAO     = 'secao'
COL_UNIDADE   = 'uo'   # coluna com código UO — ajustar se o nome for diferente
COL_GERENCIAS = None        # coluna com gerências pipe-separadas, ou None se não existir

StatementMeta(, 331b63ca-5f4b-4f4b-87dd-f3c7347fecb0, 6, Finished, Available, Finished, False)

In [5]:
# ── Leitura das fontes ────────────────────────────────────────────────────────
cols_func = [COL_EMAIL, COL_CARGO, COL_SECAO, COL_UNIDADE]
if COL_GERENCIAS:
    cols_func.append(COL_GERENCIAS)

dim_func_df      = spark.sql(f"SELECT {', '.join(cols_func)} FROM lake_gold_fatos.dbo.dim_funcionarios").toPandas()
grupos_raw_df    = spark.sql("SELECT * FROM lake_prep_siplan.dbo.rls_grupos").toPandas()
overrides_raw_df = spark.sql("SELECT * FROM lake_prep_siplan.dbo.rls_overrides").toPandas()

print(f'dim_funcionarios : {len(dim_func_df):,}')
print(f'rls_grupos       : {len(grupos_raw_df):,}')
print(f'rls_overrides    : {len(overrides_raw_df):,}')
print()
print('Colunas grupos   :', grupos_raw_df.columns.tolist())
print('Colunas overrides:', overrides_raw_df.columns.tolist())

StatementMeta(, 331b63ca-5f4b-4f4b-87dd-f3c7347fecb0, 7, Finished, Available, Finished, False)

dim_funcionarios : 9,852
rls_grupos       : 5
rls_overrides    : 2

Colunas grupos   : ['e-mail', 'perfil_acesso', 'escopo', 'unidade', 'gerencias', 'ID']
Colunas overrides: ['e-mail', 'perfil_acesso', 'escopo', 'unidade', 'gerencias', 'motivo', 'data_fim', 'ativo', 'ID']


In [6]:
# ── Classificação de dim_funcionarios ─────────────────────────────────────────
def _escopo(row):
    cargo = str(row[COL_CARGO] or '').upper()
    secao = str(row[COL_SECAO] or '').upper()
    if 'GERENTE' in cargo and 'SEDE' in secao:
        return 'GERENTE_SEDE'
    if 'SEDE' in secao:
        return 'ADMIN_CENTRAL'
    return 'UNIDADE'

def _perfil(row):
    cargo  = str(row[COL_CARGO] or '').upper()
    secao  = str(row[COL_SECAO] or '').upper()
    escopo = row['escopo']
    if 'GERENTE' in cargo:                                                     return 'GERENTE'
    if 'COORD'   in cargo:                                                     return 'COORDENADOR'
    if 'SUPERV'  in cargo:                                                     return 'SUPERVISOR'
    if 'PROG'    in secao and escopo == 'UNIDADE':                             return 'PROGRAMADOR'
    if 'SEDE'    in secao and any(k in cargo for k in ('TÉCNIC', 'ESPECIAL')): return 'ASSISTENTE'
    return 'GERAL'

oficial_df = dim_func_df.copy()
oficial_df['escopo']        = oficial_df.apply(_escopo, axis=1)
oficial_df['perfil_acesso'] = oficial_df.apply(_perfil, axis=1)
oficial_df['unidade'] = oficial_df.apply(
    lambda r: str(r[COL_UNIDADE]) if r['escopo'] == 'UNIDADE' and pd.notna(r[COL_UNIDADE]) else 'all',
    axis=1,
)
oficial_df['gerencias'] = oficial_df.apply(
    lambda r: (str(r[COL_GERENCIAS]) if COL_GERENCIAS and pd.notna(r.get(COL_GERENCIAS)) else 'all')
              if r['escopo'] == 'GERENTE_SEDE' else 'all',
    axis=1,
)
oficial_df['fonte'] = 'OFICIAL'
oficial_df = oficial_df.rename(columns={COL_EMAIL: 'email'})
oficial_df = oficial_df[['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias', 'fonte']]

print(oficial_df['perfil_acesso'].value_counts().to_string())

StatementMeta(, 331b63ca-5f4b-4f4b-87dd-f3c7347fecb0, 8, Finished, Available, Finished, False)

perfil_acesso
GERAL          6855
PROGRAMADOR    2037
COORDENADOR     395
ASSISTENTE      268
SUPERVISOR      153
GERENTE         144


In [7]:
def _clean_unidade(val):
    s = str(val).strip() if pd.notna(val) else 'all'
    if s in ('', 'nan'):
        return 'all'
    try:
        return str(int(float(s)))  # '82.0' → '82'
    except ValueError:
        return s  # 'all' ou pipe-separado: mantém para explodir depois

# ── Preparar rls_grupos ───────────────────────────────────────────────────────
grupos_df = grupos_raw_df.rename(columns={'e-mail': 'email'})
grupos_df = grupos_df[['email', 'perfil_acesso', 'escopo', 'unidade', 'gerencias']].copy()
grupos_df['unidade'] = grupos_df['unidade'].apply(_clean_unidade)
# unidade pipe-separada ('68|77') → múltiplas linhas ('68', '77')
grupos_df = grupos_df.assign(unidade=grupos_df['unidade'].str.split('|')).explode('unidade')
grupos_df['unidade'] = grupos_df['unidade'].str.strip()
grupos_df['fonte'] = 'GRUPO'

# ── Preparar rls_overrides (filtrar ativos e dentro da validade) ──────────────
ov = overrides_raw_df.rename(columns={'e-mail': 'email'})
ov = ov[['email', 'perfil_acesso', 'escopo', 'unidade', 'gerencias', 'data_fim', 'ativo']].copy()
ov['unidade'] = ov['unidade'].apply(_clean_unidade)
hoje = pd.Timestamp(date.today())
ov['data_fim'] = pd.to_datetime(ov['data_fim'], errors='coerce')
ov['ativo']    = ov['ativo'].astype(str).str.upper().isin(['TRUE', '1', 'SIM', 'YES'])
overrides_df = ov[
    ov['ativo'] & (ov['data_fim'].isna() | (ov['data_fim'] >= hoje))
][['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias']].copy()
overrides_df['fonte'] = 'OVERRIDE'

print(f'grupos_df    : {len(grupos_df)}')
print(f'overrides_df : {len(overrides_df)}')

StatementMeta(, 331b63ca-5f4b-4f4b-87dd-f3c7347fecb0, 9, Finished, Available, Finished, False)

grupos_df    : 6
overrides_df : 2


In [8]:
# ── Combinar e gravar ─────────────────────────────────────────────────────────
emails_grupos = set(grupos_df['email'].str.strip().str.lower())
base_df = oficial_df[~oficial_df['email'].str.strip().str.lower().isin(emails_grupos)].copy()

dim_rls_df = pd.concat([
    base_df,
    grupos_df[['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias', 'fonte']],
    overrides_df,
], ignore_index=True)

dim_rls_df['email'] = dim_rls_df['email'].str.strip().str.lower()
dim_rls_df = dim_rls_df.drop_duplicates()

# Garante que colunas string não tenham float NaN (causa falha no Arrow/Spark)
str_cols = ['email', 'escopo', 'perfil_acesso', 'unidade', 'gerencias', 'fonte']
dim_rls_df[str_cols] = dim_rls_df[str_cols].fillna('').astype(str)

print(f'dim_rls_usuarios: {len(dim_rls_df):,} registros')
print()
print(dim_rls_df['perfil_acesso'].value_counts().to_string())
print()
print(dim_rls_df['escopo'].value_counts().to_string())
print()
print(dim_rls_df['fonte'].value_counts().to_string())

spark.createDataFrame(dim_rls_df) \
     .write.mode('overwrite') \
     .option('overwriteSchema', 'true') \
     .saveAsTable('lake_gold_fatos.dbo.dim_rls_usuarios')

print()
print('Gravado: lake_gold_fatos.dbo.dim_rls_usuarios')

StatementMeta(, 331b63ca-5f4b-4f4b-87dd-f3c7347fecb0, 10, Finished, Available, Finished, False)

dim_rls_usuarios: 9,852 registros

perfil_acesso
GERAL          6850
PROGRAMADOR    2038
COORDENADOR     396
ASSISTENTE      265
SUPERVISOR      153
GERENTE         144
DADOS_GEDES       6

escopo
UNIDADE          8749
ADMIN_CENTRAL    1053
GERENTE_SEDE       50

fonte
OFICIAL     9844
GRUPO          6
OVERRIDE       2

Gravado: lake_gold_fatos.dbo.dim_rls_usuarios
